In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("execution_anomaly_features.csv")

print("Shape:", df.shape)
print("\nFinancial years:")
print(df["financial_year"].value_counts().sort_index())

execution_features = [
    "sanction_to_first_payment_days",
    "sanction_to_completion_days",
    "payment_duration_days",
    "payment_to_sanction_ratio",
    "completion_to_sanction_ratio",
    "has_only_in_progress_payments",
    "has_mixed_payment_status",
    "payment_count_percentile",
    "high_payment_count_indicator",
    "completed_without_payment_indicator",
    "payment_exceeds_sanction_indicator",
    "peer_p90_completion_duration",
    "long_execution_indicator"
]

print("\nMissing values:")
print(df[execution_features].isna().sum())

print("\nIndicator distributions:")
indicator_features = [
    "has_only_in_progress_payments",
    "has_mixed_payment_status",
    "high_payment_count_indicator",
    "completed_without_payment_indicator",
    "payment_exceeds_sanction_indicator",
    "long_execution_indicator"
]

for col in indicator_features:
    print(f"\n{col}")
    print(df[col].value_counts(dropna=False))

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_41372\4201728152.py:4: DtypeWarning: Columns (0: completion_date) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("execution_anomaly_features.csv")


Shape: (98755, 33)

Financial years:
financial_year
2023-2024     2953
2024-2025    17921
2025-2026    57865
2026-2027    20016
Name: count, dtype: int64

Missing values:
sanction_to_first_payment_days         26868
sanction_to_completion_days            54448
payment_duration_days                  26868
payment_to_sanction_ratio                  0
completion_to_sanction_ratio           54551
has_only_in_progress_payments              0
has_mixed_payment_status                   0
payment_count_percentile               26875
high_payment_count_indicator               0
completed_without_payment_indicator        0
payment_exceeds_sanction_indicator         0
peer_p90_completion_duration               9
long_execution_indicator                   0
dtype: int64

Indicator distributions:

has_only_in_progress_payments
has_only_in_progress_payments
0    96799
1     1956
Name: count, dtype: int64

has_mixed_payment_status
has_mixed_payment_status
0    98158
1      597
Name: count, dtype: int

In [2]:
execution_numeric = [
    "sanction_to_first_payment_days",
    "sanction_to_completion_days",
    "payment_duration_days",
    "completion_to_sanction_ratio",
    "peer_p90_completion_duration"
]

print("EXECUTION FEATURE SUMMARY\n")

print(df[execution_numeric].describe(
    percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
).T)

print("\n\nEXECUTION INDICATOR COUNTS\n")

execution_indicators = [
    "long_execution_indicator",
    "completed_without_payment_indicator",
    "payment_exceeds_sanction_indicator"
]

for col in execution_indicators:
    print(f"\n{col}")
    print(df[col].value_counts())

EXECUTION FEATURE SUMMARY

                                  count        mean         std         min  \
sanction_to_first_payment_days  71887.0  104.359634  107.166334    0.000000   
sanction_to_completion_days     44307.0  182.371950  144.580898    0.000000   
payment_duration_days           71887.0   35.733304   83.089686    0.000000   
completion_to_sanction_ratio    44204.0    0.994102    0.039138    0.090056   
peer_p90_completion_duration    98746.0  386.774103   23.157224  269.000000   

                                  50%    75%    90%    95%    99%    max  
sanction_to_first_payment_days   72.0  162.0  255.0  316.0  450.0  900.0  
sanction_to_completion_days     154.0  265.0  385.0  442.0  634.0  948.0  
payment_duration_days             0.0    0.0  144.0  221.0  371.0  933.0  
completion_to_sanction_ratio      1.0    1.0    1.0    1.0    1.0    1.0  
peer_p90_completion_duration    376.0  376.0  431.0  431.0  431.0  521.6  


EXECUTION INDICATOR COUNTS


long_execution_in

In [3]:
print("WORK STATUS DISTRIBUTION")
print(df["work_status"].value_counts(dropna=False))

print("\n\nWORK STATUS BY FINANCIAL YEAR")
print(
    pd.crosstab(
        df["financial_year"],
        df["work_status"],
        margins=True
    )
)

print("\n\nLONG EXECUTION BY FINANCIAL YEAR")
print(
    df.groupby("financial_year")["long_execution_indicator"]
      .agg(["sum", "count", "mean"])
)

print("\n\nCOMPLETED WITHOUT PAYMENT BY FINANCIAL YEAR")
print(
    df.groupby("financial_year")["completed_without_payment_indicator"]
      .agg(["sum", "count", "mean"])
)

WORK STATUS DISTRIBUTION
work_status
Physical Inspection         44162
Sanction                    24182
Vendor Identification       14405
Work partially Completed     9932
Work Completed               5152
Time Estimation               922
Name: count, dtype: int64


WORK STATUS BY FINANCIAL YEAR
work_status     Physical Inspection  Sanction  Time Estimation  \
financial_year                                                   
2023-2024                      2457        70                6   
2024-2025                     12881      1053               43   
2025-2026                     26629     11055              517   
2026-2027                      2195     12004              356   
All                           44162     24182              922   

work_status     Vendor Identification  Work Completed  \
financial_year                                          
2023-2024                         209             110   
2024-2025                        1608            1207   
2025-2026 

In [4]:
# Compare execution durations by work status

duration_cols = [
    "sanction_to_first_payment_days",
    "sanction_to_completion_days",
    "payment_duration_days"
]

print("EXECUTION DURATIONS BY WORK STATUS\n")

for status in df["work_status"].dropna().unique():
    subset = df[df["work_status"] == status]

    print(f"\n{'='*60}")
    print(f"STATUS: {status}")
    print(f"WORKS: {len(subset)}")

    print(
        subset[duration_cols]
        .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
        .T[
            ["count", "mean", "50%", "75%", "90%", "95%", "99%", "max"]
        ]
    )

EXECUTION DURATIONS BY WORK STATUS


STATUS: Physical Inspection
WORKS: 44162
                                  count        mean    50%    75%    90%  \
sanction_to_first_payment_days  43993.0  108.191462   80.0  165.0  255.0   
sanction_to_completion_days     39155.0  187.813536  161.0  272.0  392.6   
payment_duration_days           43993.0   33.177733    0.0    0.0  132.0   

                                  95%    99%    max  
sanction_to_first_payment_days  312.0  430.0  880.0  
sanction_to_completion_days     444.0  634.0  948.0  
payment_duration_days           210.0  342.0  750.0  

STATUS: Sanction
WORKS: 24182
                                count       mean   50%    75%    90%    95%  \
sanction_to_first_payment_days    3.0  83.333333  66.0  103.5  126.0  133.5   
sanction_to_completion_days       0.0        NaN   NaN    NaN    NaN    NaN   
payment_duration_days             3.0  14.333333   0.0   21.5   34.4   38.7   

                                   99%    max  
sanct

In [5]:
# Completed-work execution analysis

completed = df[df["work_status"] == "Work Completed"].copy()

print("COMPLETED WORKS BY YEAR")
print(completed["financial_year"].value_counts().sort_index())

print("\n\nCOMPLETION DURATION BY YEAR")
print(
    completed.groupby("financial_year")["sanction_to_completion_days"]
    .describe(percentiles=[0.50, 0.75, 0.90, 0.95, 0.99])
    [["count", "mean", "50%", "75%", "90%", "95%", "99%", "max"]]
)

print("\n\nLONG EXECUTION AMONG COMPLETED WORKS")
print(
    completed.groupby("financial_year")["long_execution_indicator"]
    .agg(["sum", "count", "mean"])
)

COMPLETED WORKS BY YEAR
financial_year
2023-2024     110
2024-2025    1207
2025-2026    3743
2026-2027      92
Name: count, dtype: int64


COMPLETION DURATION BY YEAR
                 count        mean    50%    75%    90%    95%     99%    max
financial_year                                                               
2023-2024        110.0  458.672727  501.5  692.0  778.8  809.4  854.00  878.0
2024-2025       1207.0  215.183927  206.0  349.0  416.4  448.7  551.52  651.0
2025-2026       3743.0  110.934544   95.0  179.0  247.0  280.0  350.00  399.0
2026-2027         92.0   12.021739    7.0   18.0   34.7   35.0   37.09   38.0


LONG EXECUTION AMONG COMPLETED WORKS
                sum  count      mean
financial_year                      
2023-2024        60    110  0.545455
2024-2025       200   1207  0.165700
2025-2026         5   3743  0.001336
2026-2027         0     92  0.000000


In [6]:
# Examine peer completion-duration thresholds by financial year

completed = df[
    df["sanction_to_completion_days"].notna()
].copy()

print("PEER P90 COMPLETION THRESHOLD BY YEAR\n")

print(
    completed.groupby("financial_year")[
        "peer_p90_completion_duration"
    ].describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )[
        ["count", "mean", "50%", "75%", "90%", "max"]
    ]
)

print("\n\nACTUAL COMPLETION DURATION vs PEER THRESHOLD\n")

completed["duration_vs_peer_p90"] = (
    completed["sanction_to_completion_days"]
    / completed["peer_p90_completion_duration"]
)

print(
    completed.groupby("financial_year")[
        "duration_vs_peer_p90"
    ].describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )[
        ["count", "mean", "50%", "75%", "90%", "95%", "99%", "max"]
    ]
)

PEER P90 COMPLETION THRESHOLD BY YEAR

                  count        mean    50%    75%    90%    max
financial_year                                                 
2023-2024        2558.0  431.202502  431.0  431.0  431.0  521.6
2024-2025       13771.0  388.591489  376.0  376.0  431.0  521.6
2025-2026       27192.0  384.043807  376.0  376.0  431.0  521.6
2026-2027         781.0  388.656594  376.0  376.0  431.0  436.0


ACTUAL COMPLETION DURATION vs PEER THRESHOLD

                  count      mean       50%       75%       90%       95%  \
financial_year                                                              
2023-2024        2558.0  0.762740  0.584687  1.099768  1.703016  1.769026   
2024-2025       13771.0  0.613753  0.561170  0.900232  1.127660  1.255319   
2025-2026       27192.0  0.379197  0.332447  0.555851  0.773936  0.952128   
2026-2027         781.0  0.114077  0.083527  0.194149  0.265957  0.284574   

                     99%       max  
financial_year               

In [7]:
# Create historical baseline using only pre-2026-27 data

historical = df[
    df["financial_year"].isin([
        "2023-2024",
        "2024-2025",
        "2025-2026"
    ])
].copy()

test_year = df[
    df["financial_year"] == "2026-2027"
].copy()

# Only completed works can be evaluated for completion duration
historical_completed = historical[
    historical["sanction_to_completion_days"].notna()
].copy()

test_completed = test_year[
    test_year["sanction_to_completion_days"].notna()
].copy()

print("Historical works:", len(historical))
print("Historical completed works:", len(historical_completed))
print("2026-27 works:", len(test_year))
print("2026-27 completed works:", len(test_completed))

# Historical peer P90
historical_peer_p90 = (
    historical_completed
    .groupby(["house", "work_category"])["sanction_to_completion_days"]
    .quantile(0.90)
    .reset_index(name="historical_peer_p90_completion_days")
)

# Apply historical baseline to 2026-27 completed works
test_completed = test_completed.merge(
    historical_peer_p90,
    on=["house", "work_category"],
    how="left"
)

test_completed["duration_vs_historical_p90"] = (
    test_completed["sanction_to_completion_days"]
    / test_completed["historical_peer_p90_completion_days"]
)

print("\nHistorical peer baseline coverage:")
print(
    test_completed["historical_peer_p90_completion_days"]
    .notna()
    .value_counts()
)

print("\n2026-27 completed works vs historical peer P90:")
print(
    test_completed["duration_vs_historical_p90"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)

Historical works: 78739
Historical completed works: 43526
2026-27 works: 20016
2026-27 completed works: 781

Historical peer baseline coverage:
historical_peer_p90_completion_days
True    781
Name: count, dtype: int64

2026-27 completed works vs historical peer P90:
count    781.000000
mean       0.113493
std        0.099398
min        0.000000
50%        0.083141
75%        0.193122
90%        0.264550
95%        0.283069
99%        0.355026
max        0.399471
Name: duration_vs_historical_p90, dtype: float64


In [8]:
print("TOP 20 LONGEST 2026-27 COMPLETED WORKS\n")

cols = [
    "work_id",
    "work_category",
    "work_status",
    "sanction_to_completion_days",
    "historical_peer_p90_completion_days",
    "duration_vs_historical_p90",
    "sanction_amount",
    "total_disbursed_amount"
]

print(
    test_completed[
        cols
    ]
    .sort_values(
        "duration_vs_historical_p90",
        ascending=False
    )
    .head(20)
    .to_string(index=False)
)

TOP 20 LONGEST 2026-27 COMPLETED WORKS

                    work_id         work_category         work_status  sanction_to_completion_days  historical_peer_p90_completion_days  duration_vs_historical_p90  sanction_amount  total_disbursed_amount
WS/MP18042/2026-2027/260755         Normal/Others Physical Inspection                        151.0                                378.0                    0.399471         493420.0                489900.0
  WS/MP524/2026-2027/271582         Normal/Others Physical Inspection                        150.0                                378.0                    0.396825         500000.0                500000.0
WS/MP18042/2026-2027/259762         Normal/Others Physical Inspection                        150.0                                378.0                    0.396825         697438.0                697438.0
WS/MP18163/2026-2027/272336         Normal/Others Physical Inspection                        149.0                                378.0     

In [9]:
# Check consistency between work_status and completion_date

has_completion = df["completion_date"].notna()

print("COMPLETION DATE vs WORK STATUS\n")

print("Works with completion_date:")
print(has_completion.sum())

print("\nWorks marked 'Work Completed':")
print((df["work_status"] == "Work Completed").sum())

print("\nWork Completed BUT no completion_date:")
print(
    ((df["work_status"] == "Work Completed") & (~has_completion)).sum()
)

print("\nCompletion date exists BUT status is NOT 'Work Completed':")
print(
    (has_completion & (df["work_status"] != "Work Completed")).sum()
)

print("\nBreakdown of completion_date exists by work_status:")
print(
    pd.crosstab(
        df["work_status"],
        has_completion,
        margins=True
    )
)

COMPLETION DATE vs WORK STATUS

Works with completion_date:
44307

Works marked 'Work Completed':
5152

Work Completed BUT no completion_date:
0

Completion date exists BUT status is NOT 'Work Completed':
39155

Breakdown of completion_date exists by work_status:
completion_date           False   True    All
work_status                                  
Physical Inspection        5007  39155  44162
Sanction                  24182      0  24182
Time Estimation             922      0    922
Vendor Identification     14405      0  14405
Work Completed                0   5152   5152
Work partially Completed   9932      0   9932
All                       54448  44307  98755


In [10]:
# ==========================================
# EXECUTION ANOMALY — HISTORICAL BASELINE
# ==========================================

# Historical years only
historical = df[
    df["financial_year"].isin([
        "2023-2024",
        "2024-2025",
        "2025-2026"
    ])
].copy()

current = df[
    df["financial_year"] == "2026-2027"
].copy()

# Completion-observable works
historical_completed = historical[
    historical["completion_date"].notna()
].copy()

current_completed = current[
    current["completion_date"].notna()
].copy()

print("Historical completed/observable works:", len(historical_completed))
print("2026-27 completed/observable works:", len(current_completed))

# Historical peer P90
historical_peer_p90 = (
    historical_completed
    .groupby(["house", "work_category"])
    ["sanction_to_completion_days"]
    .quantile(0.90)
    .reset_index(name="historical_peer_p90_days")
)

# Merge baseline onto current-year completed works
current_completed = current_completed.merge(
    historical_peer_p90,
    on=["house", "work_category"],
    how="left"
)

# Ratio: actual duration / historical peer P90
current_completed["duration_vs_peer_p90"] = (
    current_completed["sanction_to_completion_days"]
    / current_completed["historical_peer_p90_days"]
)

# Long execution rule
current_completed["long_execution_flag"] = (
    current_completed["duration_vs_peer_p90"] > 1.0
).astype(int)

print("\nHistorical peer baseline coverage:")
print(
    current_completed["historical_peer_p90_days"]
    .notna()
    .value_counts()
)

print("\nLong execution flags:")
print(
    current_completed["long_execution_flag"]
    .value_counts()
)

print("\nDuration ratio:")
print(
    current_completed["duration_vs_peer_p90"]
    .describe(
        percentiles=[0.50, 0.75, 0.90, 0.95, 0.99]
    )
)


Historical completed/observable works: 43526
2026-27 completed/observable works: 781

Historical peer baseline coverage:
historical_peer_p90_days
True    781
Name: count, dtype: int64

Long execution flags:
long_execution_flag
0    781
Name: count, dtype: int64

Duration ratio:
count    781.000000
mean       0.113493
std        0.099398
min        0.000000
50%        0.083141
75%        0.193122
90%        0.264550
95%        0.283069
99%        0.355026
max        0.399471
Name: duration_vs_peer_p90, dtype: float64


In [11]:
# ==========================================
# EXECUTION ANOMALY — CONSISTENCY CHECKS
# ==========================================

current["completed_without_payment_flag"] = (
    current["completion_date"].notna()
    & (current["payment_count"] == 0)
).astype(int)

current["payment_exceeds_sanction_flag"] = (
    current["total_disbursed_amount"] > current["sanction_amount"] + 1.0
).astype(int)

print("2026-27 EXECUTION CONSISTENCY FLAGS\n")

print(
    "Completed without payment:",
    current["completed_without_payment_flag"].sum()
)

print(
    "Payment exceeds sanction:",
    current["payment_exceeds_sanction_flag"].sum()
)

# Show the actual cases
flagged = current[
    (current["completed_without_payment_flag"] == 1) |
    (current["payment_exceeds_sanction_flag"] == 1)
].copy()

cols = [
    "work_id",
    "work_category",
    "work_status",
    "sanction_date",
    "completion_date",
    "sanction_amount",
    "total_disbursed_amount",
    "payment_count",
    "completed_without_payment_flag",
    "payment_exceeds_sanction_flag"
]

print("\nFLAGGED 2026-27 WORKS:\n")

print(
    flagged[cols]
    .sort_values(
        ["payment_exceeds_sanction_flag",
         "completed_without_payment_flag"],
        ascending=False
    )
    .to_string(index=False)
)

2026-27 EXECUTION CONSISTENCY FLAGS

Completed without payment: 11
Payment exceeds sanction: 0

FLAGGED 2026-27 WORKS:

                    work_id work_category         work_status sanction_date completion_date  sanction_amount  total_disbursed_amount  payment_count  completed_without_payment_flag  payment_exceeds_sanction_flag
  WS/MP347/2026-2027/196522 Normal/Others Physical Inspection    2026-05-20      2026-08-18         300000.0                     0.0              0                               1                              0
  WS/MP347/2026-2027/196523 Normal/Others Physical Inspection    2026-05-20      2026-08-18         300000.0                     0.0              0                               1                              0
  WS/MP347/2026-2027/196524 Normal/Others Physical Inspection    2026-05-20      2026-08-18         200000.0                     0.0              0                               1                              0
  WS/MP440/2026-2027/256252 Normal/O

In [2]:
print("Variables available:")
print([x for x in globals().keys() if not x.startswith("_")])

Variables available:
['In', 'Out', 'get_ipython', 'exit', 'quit', 'open']


In [1]:
# ==========================================
# FINAL EXECUTION ANOMALY MODULE
# ==========================================

execution_results = current.copy()

# ------------------------------------------
# 1. Historical peer P90
# ------------------------------------------

historical = df[
    df["financial_year"].isin([
        "2023-2024",
        "2024-2025",
        "2025-2026"
    ])
].copy()

historical_completed = historical[
    historical["completion_date"].notna()
].copy()

historical_peer_p90 = (
    historical_completed
    .groupby(["house", "work_category"])
    ["sanction_to_completion_days"]
    .quantile(0.90)
    .reset_index(name="historical_peer_p90_days")
)

execution_results = execution_results.merge(
    historical_peer_p90,
    on=["house", "work_category"],
    how="left"
)

# ------------------------------------------
# 2. Duration comparison
# ------------------------------------------

execution_results["duration_vs_peer_p90"] = np.where(
    execution_results["historical_peer_p90_days"] > 0,
    execution_results["sanction_to_completion_days"]
    / execution_results["historical_peer_p90_days"],
    np.nan
)

execution_results["long_execution_flag"] = (
    execution_results["completion_date"].notna()
    & execution_results["historical_peer_p90_days"].notna()
    & (
        execution_results["sanction_to_completion_days"]
        > execution_results["historical_peer_p90_days"]
    )
).astype(int)

# ------------------------------------------
# 3. Completed without payment
# ------------------------------------------

execution_results["completed_without_payment_flag"] = (
    execution_results["completion_date"].notna()
    & (execution_results["payment_count"] == 0)
).astype(int)

# ------------------------------------------
# 4. Payment exceeds sanction
# ------------------------------------------

execution_results["payment_exceeds_sanction_flag"] = (
    execution_results["total_disbursed_amount"]
    > execution_results["sanction_amount"] + 1.0
).astype(int)

# ------------------------------------------
# 5. RULE-BASED EXECUTION SCORE
# ------------------------------------------

execution_results["execution_anomaly_score"] = (
    execution_results["long_execution_flag"]
    + execution_results["completed_without_payment_flag"]
    + execution_results["payment_exceeds_sanction_flag"]
)

# ------------------------------------------
# 6. Final anomaly flag
# ------------------------------------------

execution_results["execution_anomaly_flag"] = (
    execution_results["execution_anomaly_score"] > 0
).astype(int)

# ------------------------------------------
# 7. Review level
# ------------------------------------------

execution_results["execution_review_level"] = np.select(
    [
        execution_results["execution_anomaly_score"] >= 2,
        execution_results["execution_anomaly_score"] == 1
    ],
    [
        "HIGH_REVIEW",
        "REVIEW"
    ],
    default="NORMAL"
)

# ------------------------------------------
# 8. Explainable reason
# ------------------------------------------

def build_execution_reason(row):

    reasons = []

    if row["long_execution_flag"] == 1:
        reasons.append(
            "Completion duration exceeds historical peer benchmark"
        )

    if row["completed_without_payment_flag"] == 1:
        reasons.append(
            "Completed work has no recorded payment"
        )

    if row["payment_exceeds_sanction_flag"] == 1:
        reasons.append(
            "Payment exceeds sanctioned amount"
        )

    if not reasons:
        return "No execution anomaly detected"

    return "; ".join(reasons)


execution_results["execution_anomaly_reason"] = (
    execution_results.apply(
        build_execution_reason,
        axis=1
    )
)

# ------------------------------------------
# 9. FINAL SUMMARY
# ------------------------------------------

print("=" * 70)
print("FINAL EXECUTION ANOMALY SUMMARY")
print("=" * 70)

print("Total 2026-27 works:",
      len(execution_results))

print("Long execution cases:",
      int(execution_results["long_execution_flag"].sum()))

print("Completed without payment:",
      int(execution_results["completed_without_payment_flag"].sum()))

print("Payment exceeds sanction:",
      int(execution_results["payment_exceeds_sanction_flag"].sum()))

print("\nExecution anomaly cases:",
      int(execution_results["execution_anomaly_flag"].sum()))

print("\nReview distribution:")
print(
    execution_results["execution_review_level"]
    .value_counts()
)

print("\nScore distribution:")
print(
    execution_results["execution_anomaly_score"]
    .value_counts()
    .sort_index()
)

NameError: name 'current' is not defined

In [16]:
# ==========================================
# EXPORT CORRECTED EXECUTION MODULE
# ==========================================

execution_export_cols = [
    "work_id",
    "house",
    "mp_key",
    "state",
    "ida",
    "work_category",
    "work_title",
    "financial_year",
    "work_status",
    "sanction_date",
    "completion_date",
    "sanction_amount",
    "total_disbursed_amount",
    "payment_count",
    "successful_payment_count",
    "in_progress_payment_count",
    "sanction_to_first_payment_days",
    "sanction_to_completion_days",
    "payment_duration_days",
    "payment_to_sanction_ratio",
    "completion_to_sanction_ratio",
    "historical_peer_p90_days",
    "duration_vs_peer_p90",
    "long_execution_flag",
    "completed_without_payment_flag",
    "payment_exceeds_sanction_flag",
    "consistency_review_flag",
    "execution_anomaly_score",
    "execution_anomaly_flag",
    "execution_review_level",
    "execution_anomaly_reason"
]

execution_final = execution_results[execution_export_cols].copy()

execution_final.to_csv(
    "execution_anomaly_results_2026_27.csv",
    index=False
)

print("CORRECTED EXECUTION MODULE EXPORTED")
print("=" * 55)
print("Rows:", len(execution_final))
print("Columns:", len(execution_final.columns))
print("File: execution_anomaly_results_2026_27.csv")

print("\nTrue execution anomalies:",
      execution_final["execution_anomaly_flag"].sum())

print("Consistency review cases:",
      execution_final["consistency_review_flag"].sum())

CORRECTED EXECUTION MODULE EXPORTED
Rows: 20016
Columns: 31
File: execution_anomaly_results_2026_27.csv

True execution anomalies: 0
Consistency review cases: 11


In [13]:
# ==========================================
# FINAL EXECUTION ANOMALY CASES
# ==========================================

review_cases = execution_results[
    execution_results["execution_review_level"] != "NORMAL"
].copy()

review_cols = [
    "work_id",
    "financial_year",
    "work_category",
    "work_status",
    "sanction_date",
    "completion_date",
    "sanction_amount",
    "total_disbursed_amount",
    "payment_count",
    "execution_anomaly_score",
    "execution_review_level",
    "execution_anomaly_reason"
]

print("FINAL EXECUTION REVIEW CASES")
print("=" * 70)

print(
    review_cases[review_cols]
    .sort_values("execution_anomaly_score", ascending=False)
    .to_string(index=False)
)

FINAL EXECUTION REVIEW CASES
                    work_id financial_year work_category         work_status sanction_date completion_date  sanction_amount  total_disbursed_amount  payment_count  execution_anomaly_score execution_review_level               execution_anomaly_reason
  WS/MP347/2026-2027/196522      2026-2027 Normal/Others Physical Inspection    2026-05-20      2026-08-18         300000.0                     0.0              0                        2                 REVIEW Completed work has no recorded payment
  WS/MP347/2026-2027/196523      2026-2027 Normal/Others Physical Inspection    2026-05-20      2026-08-18         300000.0                     0.0              0                        2                 REVIEW Completed work has no recorded payment
  WS/MP347/2026-2027/196524      2026-2027 Normal/Others Physical Inspection    2026-05-20      2026-08-18         200000.0                     0.0              0                        2                 REVIEW Completed 

In [14]:
# ==========================================
# EXPORT FINAL EXECUTION ANOMALY RESULTS
# ==========================================

# Select useful columns for the final output
execution_export_cols = [
    "work_id",
    "house",
    "mp_key",
    "state",
    "ida",
    "work_category",
    "work_title",
    "financial_year",
    "work_status",
    "sanction_date",
    "completion_date",
    "sanction_amount",
    "total_disbursed_amount",
    "payment_count",
    "successful_payment_count",
    "in_progress_payment_count",
    "sanction_to_first_payment_days",
    "sanction_to_completion_days",
    "payment_duration_days",
    "payment_to_sanction_ratio",
    "completion_to_sanction_ratio",
    "historical_peer_p90_days",
    "duration_vs_peer_p90",
    "long_execution_flag",
    "completed_without_payment_flag",
    "payment_exceeds_sanction_flag",
    "execution_anomaly_score",
    "execution_review_level",
    "execution_anomaly_reason"
]

execution_final = execution_results[execution_export_cols].copy()

# Save
execution_final.to_csv(
    "execution_anomaly_results_2026_27.csv",
    index=False
)

print("EXECUTION ANOMALY MODULE EXPORTED")
print("=" * 50)
print("Rows:", len(execution_final))
print("Columns:", len(execution_final.columns))
print("File: execution_anomaly_results_2026_27.csv")

print("\nReview distribution:")
print(
    execution_final["execution_review_level"]
    .value_counts()
)

print("\nAnomaly cases:")
print(
    (execution_final["execution_review_level"] != "NORMAL").sum()
)

EXECUTION ANOMALY MODULE EXPORTED
Rows: 20016
Columns: 29
File: execution_anomaly_results_2026_27.csv

Review distribution:
execution_review_level
NORMAL    20005
REVIEW       11
Name: count, dtype: int64

Anomaly cases:
11
